In [77]:
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import adfuller

from tqdm import tqdm_notebook
from itertools import product
from typing import Union

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

In [78]:
# read in the merged station datasets
dfs = {}
for index in range(0, 6) :
  df = pd.read_csv('merged_' + str(index + 1) + '.dat', sep=",", parse_dates=["Date"], index_col="Date")
  dfs['Station' + str(index + 1)] = df


In [79]:
#drop these two features because they mess up station 4. Temporary solution
for key in dfs.keys():
  dfs[key].drop(['SWC_50'], axis = 1, inplace = True)
  dfs[key].drop(['T_50'], axis = 1, inplace = True)


In [80]:
#drop flag feature and null values

for station, df in dfs.items() :
  # drop the Flag feature (it is not relevant to our soil data and can cause our ML model to pick on unneccesary patterns)
  df = df.drop('Flag', axis = 1)
  # rename Ppt columns to identify whether the precipitation was recorded as a part of soil data or meteorological data
  df_new = df.rename(columns = {'Ppt_x' : 'Ppt_soil', 'Ppt_y': 'Ppt_met'})
  df_new.dropna(inplace=True)
  dfs[station] = df_new

In [81]:
#Vectorize wind
for station, df in dfs.items() :
  # convert wind velocity and wind direction to a wind vector
  wv = df.pop('Windspeed')

  # Convert to radians.
  wd_rad = df.pop('Winddirection')*np.pi / 180

  # Calculate the wind x and y components.
  df['Wx'] = wv*np.cos(wd_rad)
  df['Wy'] = wv*np.sin(wd_rad)
  dfs[station] = df

In [ ]:
# Remove periodicity in time data (remove daily and yearly periodicity)
day = 24*60*60
year = (365.2425)*day

for station, df in dfs.items() :
  timestamp_s = (df.index).map(pd.Timestamp.timestamp)

  df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
  df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
  df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
  df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

  dfs[station] = df

In [ ]:
# create one standard metric for PPT
for key in dfs.keys():
  dfs[key]['PPT'] = (dfs[key].pop('Ppt_soil') + dfs[key].pop('Ppt_met'))/2

In [ ]:
#Normalize all the data

for key in dfs.keys():
  dfs[key] = (dfs[key] - dfs[key].min())/(dfs[key].max()-dfs[key].min())

In [ ]:
#add geographical position data

position_dict = {"Station1": [30.3989,-98.6105],
                 "Station2": [30.4193,-98.8046],
                 "Station3": [30.4421,-98.8427],
                 "Station4": [30.4600, -98.9407],
                 "Station5": [30.2454,-98.7059],
                 "Station6": [30.2758,-98.7242]
                 }

for key in dfs.keys():
  dfs[key]["Latitude"] = position_dict[key][0]
  dfs[key]["Longitude"] = position_dict[key][1]

In [ ]:
#only use data shared all together: indexes that are none null for each station

index_union = pd.Index([])
for station, df in dfs.items():
  index_union = index_union.union(df.index)

index_int = index_union
for station, df in dfs.items():
  index_int = index_int.intersection(df.index)


print(len(index_int))


# Create Multi-Index DF for all Station Data

In [ ]:
feats = dfs['Station1'].columns.tolist()

stations = list(dfs.keys())

In [ ]:
index = index_int

cols = pd.MultiIndex.from_product([stations,feats], names = ['Station', 'Feature'])

In [ ]:
for station, df in dfs.items():
  cols_new = []
  for col in df.columns.tolist():
    cols_new.append(f"{station} {col}")
  df.columns = cols_new

In [ ]:
lis = [dfs[key].loc[index] for key in dfs.keys()]

data = pd.concat(lis, axis = 1)

data.head()

In [ ]:
df = pd.DataFrame(data = data.values, index = index, columns = cols)

# Data Split

In [ ]:
#create Df of our target Values

target_station = 'Station6'

target_names = ['SWC_5']

target_df = df[target_station][target_names]

target_df.shape

In [ ]:
#Create DF of all our data not the target

non_targets = list(dfs.keys())

non_targets.remove('Station6')

train_df = df[non_targets]

train_df.shape

train=[]
test=[]
# PPT_exog = []
# Tair_exog = []

for station, feature in df:
    
    if feature == "SWC_5" and station != 'Station6':
        train.extend(df[station].SWC_5)
    elif feature == "SWC_5":
        test.extend(df[station].SWC_5)
    
#     if feature == "PPT" and station != "Station6":
#         PPT_exog.extend(df[station].PPT)
#     if feature == "Tair" and station != "Station6":
#         Tair_exog.extend(df[station].Tair)

In [ ]:
df_train = pd.DataFrame(train[::24*30])
df_test = pd.DataFrame(test[::24*30])
df_total = pd.concat([df_train, df_test], ignore_index=True)
# df_Tair_exog = pd.DataFrame(Tair_exog[::24*30])

train_len = len(df_train)
test_len = len(df_test)

total_len = train_len + test_len

# want them seperate, one vector each statement
# train sequentially

# one per day (not month)
# adjust time to be index and use mean()
# youtube vidio under referances in git

In [ ]:
df_test

In [ ]:
# for station, feature in df:
    
#     if feature == "SWC_5":
    
#         fig, ax = plt.subplots()

#         ax.plot(df[station].SWC_5)
#         ax.set_xlabel('Date')
#         ax.set_ylabel('Soil Water Content')

#         fig.autofmt_xdate()
#         plt.tight_layout()

In [ ]:
# Define SARIMA model
from typing import Union
from tqdm import tqdm_notebook
from statsmodels.tsa.statespace.sarimax import SARIMAX

def optimize_SARIMAX(endog: Union[pd.Series, list], exog: Union[pd.Series, list], order_list: list, d: int, D: int, s: int) -> pd.DataFrame:
    
    results = []
    
    for order in tqdm_notebook(order_list):
        try: 
            model = SARIMAX(
                endog,
                exog,
                order=(order[0], d, order[1]),
                seasonal_order=(order[2], D, order[3], s),
                simple_differencing=False).fit(disp=False)
        except:
            continue
            
        aic = model.aic
        print([order, aic])
        results.append([order, model.aic])
        
    result_df = pd.DataFrame(results)
    result_df.columns = ['(p,q,P,Q)', 'AIC']
    
    #Sort in ascending order, lower AIC is better
    result_df = result_df.sort_values(by='AIC', ascending=True).reset_index(drop=True)
    
    return result_df

In [ ]:
# Define range of parameter's to check
ps = range(0, 2, 1)
qs = range(0, 2, 1)
Ps = range(0, 2, 1)
Qs = range(0, 2, 1)

order_list = list(product(ps, qs, Ps, Qs))

d = 0
D = 0
s = 12


In [ ]:
# Find best set of parameters using the AIC
SARIMA_result_df = optimize_SARIMAX(df_train, None, order_list, d, D, s)
SARIMA_result_df

In [ ]:
# Define and fit SARIMAX model
# call on each or optimize each
# N X 5 station to train (.value turns into a numpy array)
SARIMA_model = SARIMAX(df_train, order=((SARIMA_result_df.iloc[0][0][0]), 0, SARIMA_result_df.iloc[0][0][1]), seasonal_order = (SARIMA_result_df.iloc[0][0][2], 0, SARIMA_result_df.iloc[0][0][3], s), simple_differencing=False, enforce_stationarity=False)
SARIMA_model_fit = SARIMA_model.fit(disp=False)

print(SARIMA_model_fit.summary())

In [ ]:
## Forecasting 

def rolling_forecast(df: pd.DataFrame, train_len: int, horizon: int, window: int, method: str) -> list:
    
    total_len = train_len + horizon
    end_idx = train_len
    
    
    if method == 'last_season':
        pred_last_season = []
            
        for i in range(train_len, total_len, window):
            last_season = df[0][i-window:i]
            pred_last_season.extend(last_season)
            
            
        return pred_last_season
    
    elif method == 'SARIMA':
        pred_SARIMA = []
        
        for i in range(train_len, total_len, window):
            model = SARIMAX(df_train[:i], order=(SARIMA_result_df.iloc[0][0][0], 0, SARIMA_result_df.iloc[0][0][1]), seasonal_order = (SARIMA_result_df.iloc[0][0][2], 0, SARIMA_result_df.iloc[0][0][3], s), simple_differencing=False)
            res = model.fit(disp=False)
            predictions = res.get_prediction(0, i + window - 1)
            oos_pred = predictions.predicted_mean[-window:]
            pred_SARIMA.extend(oos_pred)
            
        return pred_SARIMA

In [ ]:
pred_df = df_test.copy()

In [ ]:
# Define rolling forcast structure and create baseline for model by recalling the previous season
TRAIN_LEN = train_len
HORIZON = test_len
WINDOW = test_len

pred_df['last_season'] = rolling_forecast(df_train, TRAIN_LEN, HORIZON, WINDOW, 'last_season')
pred_df.last_season

In [ ]:
# Define SARIMA Prediction Dataframe
pred_df['SARIMA'] = rolling_forecast(df_total, TRAIN_LEN, HORIZON, WINDOW, 'SARIMA')

pred_df.SARIMA

In [ ]:
# visualize predictions
fig, ax = plt.subplots()

# ax.plot(df_total)
ax.plot(df_test, 'b-', label='actual')
# ax.plot(pred_df.last_season, 'r:', label='naive seasonal')
ax.plot(pred_df.SARIMA, 'k--', label='SARIMA')
ax.set_xlabel('Months')
ax.set_ylabel('Normalized SWC_5')

fig.autofmt_xdate()
plt.tight_layout()

In [ ]:
# Evaluate 

mse=np.mean((df_test-pred_df.SARIMA)**2)
mse